In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from collections import defaultdict, deque
import random
import math
import os
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# constants
ROAD_LENGTH = 100 * 1000  # total road length in meters
RSU_SPACING = 1000   # 1 km between RSUs
RSU_COUNT = (ROAD_LENGTH // RSU_SPACING) + 1
RSU_RADIUS = 500     # coverage radius in meters
CACHE_SIZE_RSU = 200 # MBs
CACHE_SIZE_VEHICLE = 200 # MBs
VEHICLE_COUNT = 1510
VEHICLE_SPEED = 15   # m/s
USERS_PER_VEHICLE = 4

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

ratings_dataset = pd.read_csv("ratings.dat", sep="::", engine="python",
                      names=["UserID", "MovieID", "Rating", "Timestamp"])

users = pd.read_csv("users.dat", sep="::", engine="python",
                    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"])

movies = pd.read_csv("movies.dat", sep="::", engine="python",
                     names=["MovieID", "Title", "Genres"], encoding="ISO-8859-1")

valid_user_ids = set(users['UserID'])
valid_movie_ids = set(movies['MovieID'])

ratings_dataset = ratings_dataset[
    ratings_dataset['UserID'].isin(valid_user_ids) &
    ratings_dataset['MovieID'].isin(valid_movie_ids)
]

ratings_dataset = ratings_dataset.sort_values("Timestamp").reset_index(drop=True)
split_idx = int(len(ratings_dataset) * 0.98)
train_ratings = ratings_dataset.iloc[:split_idx]
test_ratings = ratings_dataset.iloc[split_idx:]

user_id_to_idx = {uid: idx for idx, uid in enumerate(sorted(valid_user_ids))}
movie_id_to_idx = {mid: idx for idx, mid in enumerate(sorted(valid_movie_ids))}

num_users = len(user_id_to_idx)
num_movies = len(movie_id_to_idx)
ratings_matrix = np.zeros((num_users, num_movies))

for row in train_ratings.itertuples():
    u_idx = user_id_to_idx[row.UserID]
    m_idx = movie_id_to_idx[row.MovieID]
    ratings_matrix[u_idx, m_idx] = row.Rating

users = users.set_index("UserID").loc[sorted(valid_user_ids)].copy()
users["Gender"] = LabelEncoder().fit_transform(users["Gender"])
users["Zip-code"] = LabelEncoder().fit_transform(users["Zip-code"])
scaler = StandardScaler()
users[["Age", "Zip-code"]] = scaler.fit_transform(users[["Age", "Zip-code"]])
user_features = users[["Gender", "Age", "Occupation"]].values

movies = movies.set_index("MovieID").loc[sorted(valid_movie_ids)].copy()
all_genres = sorted(set(g for gs in movies["Genres"].str.split('|') for g in gs))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}

def build_genre_vector(genre_str):
    vec = np.zeros(len(all_genres))
    for g in genre_str.split('|'):
        if g in genre_to_idx:
            vec[genre_to_idx[g]] = 1
    return vec

movies["GenreVec"] = movies["Genres"].apply(build_genre_vector)
genre_matrix = np.vstack(movies["GenreVec"].values)

print(f"Ratings matrix shape: {ratings_matrix.shape}")
print(f"User features shape: {user_features.shape}")
print(f"Genre matrix shape: {genre_matrix.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rsus = []
for i in range(RSU_COUNT):
    rsu = {
        'id': i,
        'position': i * RSU_SPACING,
        'cache': set(),  # content IDs
    }
    rsus.append(rsu)

vehicles = []
vehicle_positions = np.linspace(0, ROAD_LENGTH, VEHICLE_COUNT, endpoint=False)

available_user_indices = np.arange(len(user_features))
np.random.shuffle(available_user_indices)

assert VEHICLE_COUNT * USERS_PER_VEHICLE <= len(available_user_indices), "Not enough users available!"

for i, pos in enumerate(vehicle_positions):
    user_start = i * USERS_PER_VEHICLE
    user_ids = available_user_indices[user_start:user_start + USERS_PER_VEHICLE].tolist()

    # Find the RSU this vehicle is within range of
    rsu_index = None
    for j, rsu in enumerate(rsus):
        if abs(pos - rsu['position']) <= RSU_RADIUS:
            rsu_index = j
            break
    assert rsu_index is not None, f"Vehicle {i} at pos {pos} not within any RSU range"

    vehicle = {
        'id': i,
        'initial_position': pos,
        'position': pos,
        'speed': VEHICLE_SPEED,
        'users': user_ids,
        'cache': set(),      # content IDs
        'initial_rsu_index': rsu_index,
        'rsu_index': rsu_index,  # index of the associated RSU
    }
    vehicles.append(vehicle)

print(f"Initialized {len(rsus)} RSUs and {len(vehicles)} vehicles.")

rsu_positions = [r['position'] for r in rsus]
vehicle_positions_plot = [v['position'] for v in vehicles]

plt.figure(figsize=(12, 1.5))
plt.scatter(rsu_positions, np.zeros_like(rsu_positions), label='RSUs', marker='^', c='red')
plt.scatter(vehicle_positions_plot, np.ones_like(vehicle_positions_plot), label='Vehicles', marker='o', c='blue', alpha=0.5)
plt.title("RSU and Vehicle Initial Placement")
plt.yticks([])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
T_train = 2.5    # seconds
T_inf = 0.5      # seconds
ROUND_DURATION = T_train + T_inf
RSU_DIAMETER = 2 * RSU_RADIUS

a1, a2, a3 = 0.4, 0.3, 0.3

def move_vehicles(vehicles):
    for v in vehicles:
        v['position'] += v['speed'] * ROUND_DURATION

def update_vehicle_rsu_assignments(vehicles, rsus):
    for v in vehicles:
        rsu_index = None
        for i, rsu in enumerate(rsus):
            if abs(v['position'] - rsu['position']) <= RSU_RADIUS:
                rsu_index = i
                break
        v['rsu_index'] = rsu_index

def calculate_staying_time(vehicle, rsu):
    Pv = abs(vehicle['position'] - (rsu['position'] - RSU_RADIUS)) # distance from entrance position
    D = RSU_DIAMETER
    return (D - Pv) / vehicle['speed']

def select_vehicles(rsu, vehicles):
    selected = []
    for v in vehicles:
        if v['rsu_index'] != rsu['id']:
            continue
        staying_time = calculate_staying_time(v, rsu)
        if staying_time > ROUND_DURATION:
            v['staying_time'] = staying_time
            selected.append(v)
    return selected

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=100):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
def train_local_model(model, user_indices, ratings_matrix_norm, epochs=1, lr=1e-2):
    device = next(model.parameters()).device
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for _ in range(epochs):
        for u in user_indices:
            input_vec = torch.FloatTensor(ratings_matrix_norm[u]).to(device)
            optimizer.zero_grad()
            output = model(input_vec)
            loss = loss_fn(output, input_vec)
            loss.backward()
            optimizer.step()

    return model.cpu()  # move back to CPU after training

In [ ]:
B = 540000            # 540 KHz
SIGMA2 = 3.98e-15  # from -114 dBm

P_V = 0.002            # W (V2V) == 3dBm
P_S = 1.0            # W (RSU) == 30dBm
P_M = 19.95          # W (MBS) == 43dBm
WIRED_R_RP = 100e6   # 100 Mbps between RSUs (RR,R')

PATHLOSS_EXP = 3.0
PL_K = 1.0
SHADOW_FADING_STD_DB = 4

def channel_gain(distance_m):
    d = max(distance_m, 1.0)
    path_loss = PL_K / (d ** PATHLOSS_EXP)
    shadow_db = np.random.normal(0, SHADOW_FADING_STD_DB)
    shadow_lin = 10 ** (shadow_db / 10)
    return path_loss * shadow_lin

def rate_v2r(d_m):
    h = channel_gain(d_m)
    return B * math.log2(1.0 + (P_S * h) / SIGMA2)

def compute_vehicle_weight(vehicle, max_vals):
    T_stay = vehicle['staying_time']
    R = rate_v2r(abs(vehicle['position'] - rsus[vehicle['rsu_index']]['position']))
    c = CACHE_SIZE_VEHICLE

    return (
        a1 * (T_stay / max_vals['T']) +
        a2 * (R / max_vals['R']) +
        a3 * (c / max_vals['C'])
    )

SERVER_LR = 0.01

def aggregate_one_upload(global_model, client_model, Wn):
    """
    ω ← ω + η * W_n * (ω_n − ω)
    """
    g = global_model.state_dict()
    s = client_model.state_dict()
    wf = float(Wn)
    for k in g.keys():
        g[k] = g[k] + SERVER_LR * wf * (s[k] - g[k])
    global_model.load_state_dict(g)
    return global_model

In [ ]:
def federated_round(rsu, vehicles, global_model, ratings_matrix_norm):
    selected_vehicles = select_vehicles(rsu, vehicles)
    if not selected_vehicles:
        return global_model  # no updates

    max_vals = {
        'T': max(v['staying_time'] for v in selected_vehicles),
        'R': 0.0,
        'C': CACHE_SIZE_VEHICLE  # same for all
    }

    for vehicle in selected_vehicles:
        R = rate_v2r(abs(vehicle['position'] - rsus[vehicle['rsu_index']]['position']))
        max_vals['R'] = max(max_vals['R'], R)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for v in selected_vehicles:
        local_model = AutoEncoder(input_dim=ratings_matrix_norm.shape[1]).to(device)
        local_model.load_state_dict(global_model.state_dict())  # download ω

        trained_model = train_local_model(
            local_model,
            v['users'],  # 4 users per vehicle
            ratings_matrix_norm
        )

        W_n = compute_vehicle_weight(v, max_vals)
        global_model = aggregate_one_upload(global_model, trained_model, W_n)

    return global_model

In [ ]:
ratings_matrix_norm = ratings_matrix / 5.0

# Initialize per-RSU model
for rsu in rsus:
    rsu['model'] = AutoEncoder(input_dim=ratings_matrix.shape[1])

num_rounds = 100
for r in range(num_rounds):
    print(f"Round {r+1}")

    for rsu in rsus:
        rsu['model'] = federated_round(rsu, vehicles, rsu['model'], ratings_matrix_norm)

    move_vehicles(vehicles)
    update_vehicle_rsu_assignments(vehicles, rsus)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def get_encoded_users(autoencoder, ratings_matrix):
    autoencoder.eval()
    with torch.no_grad():
        inputs = torch.FloatTensor(ratings_matrix)
        encoded = autoencoder.encoder(inputs)
    return encoded.numpy()

# === Encode users
encoded_users = np.zeros((num_users, 100))
with tqdm(total=VEHICLE_COUNT, desc="Encoding users") as pbar:
    for rsu in rsus:
        model = rsu['model']
        rsu_vehicles = [v for v in vehicles if v['initial_rsu_index'] == rsu['id']]

        for vehicle in rsu_vehicles:
            vehicle_ratings_norm = ratings_matrix_norm[vehicle['users']]
            encoded_users[vehicle['users']] = get_encoded_users(model, vehicle_ratings_norm)
            pbar.update(1)

def user_similarity(i_ratings, j_ratings, i_info, j_info):
    i_rated = i_ratings > 0
    j_rated = j_ratings > 0
    co_mask = i_rated & j_rated
    if co_mask.sum() == 0:
        return 0.0  # no basis to compare

    mean_i = i_ratings[i_rated].mean()
    mean_j = j_ratings[j_rated].mean()

    xi = i_ratings[co_mask] - mean_i
    xj = j_ratings[co_mask] - mean_j
    denom = (np.linalg.norm(xi) * np.linalg.norm(xj)) + 1e-8
    rating_sim = float(np.dot(xi, xj) / denom)

    info_denom = (np.linalg.norm(i_info) * np.linalg.norm(j_info)) + 1e-8
    info_sim = float(np.dot(i_info, j_info) / info_denom)

    return rating_sim * info_sim

def collect_neighbors(user_index, theta_th=0.5):
    U = ratings_matrix.shape[0]
    neighbors = []

    i_ratings = ratings_matrix[user_index]
    i_info  = np.append(user_features[user_index], encoded_users[user_index])

    for j in range(U):
        if j == user_index:
            continue

        j_ratings = ratings_matrix[j]
        j_info = np.append(user_features[j], encoded_users[j])
        theta = user_similarity(i_ratings, j_ratings, i_info, j_info)

        if theta > theta_th:
            rj = ratings_matrix[j]
            rated_mask_j = (rj > 0)
            if not np.any(rated_mask_j):
                continue
            mean_j = rj[rated_mask_j].mean()
            neighbors.append({
                "idx": j,
                "theta": float(theta),
                "mean": float(mean_j),
                "rated_mask": rated_mask_j
            })

    return neighbors

def predict_user_ratings(user_index):
    r_u = ratings_matrix[user_index]
    u_avg = r_u[r_u > 0].mean() if np.any(r_u > 0) else 0
    preds = r_u.copy()

    neighbors = collect_neighbors(user_index)

    for f in range(len(r_u)):
        if r_u[f] > 0:
            continue

        numer, denom = 0, 0
        for nb in neighbors:
            j = nb["idx"]
            if not nb["rated_mask"][f]:
                continue  # neighbor must have rated f
            r_jf = ratings_matrix[j, f]
            numer += nb["theta"] * (r_jf - nb["mean"])
            denom += nb["theta"]

        preds[f] = u_avg + (numer / (denom + 1e-8))

    return preds

In [ ]:
def compute_theme_interest(ratings_u):
    theme_counts = genre_matrix.sum(axis=0)                  # (M,)
    p_global = (theme_counts / (theme_counts.sum() + 1e-8))    # (M,)

    theme_mass_user = genre_matrix.T @ ratings_u             # (M,)
    p_user = theme_mass_user / (theme_mass_user.sum() + 1e-8)  # (M,)

    phi_u_h = np.log((p_user + 1e-8) / (p_global + 1e-8))       # (M,)


    num = genre_matrix @ phi_u_h                             # (F,)
    denom = (np.linalg.norm(genre_matrix, axis=1) *          # ||ρ_{:,f}||
             (np.linalg.norm(phi_u_h) + 1e-8) + 1e-8)
    phi_u_f = num / denom

    return phi_u_h, phi_u_f

In [ ]:
def request_probability(r_u_f, phi_u_f, ε=5.0, phi_weight=0.3):
    g_r = (r_u_f - r_u_f.min()) / (r_u_f.max() - r_u_f.min() + 1e-8)
    g_phi = (phi_u_f - phi_u_f.min()) / (phi_u_f.max() - phi_u_f.min() + 1e-8)

    p_r = np.exp(ε * g_r)
    p_phi = np.exp(ε * g_phi)

    p_r /= np.sum(p_r)
    p_phi /= np.sum(p_phi)

    return phi_weight * p_r + (1 - phi_weight) * p_phi

In [ ]:
def compute_vehicle_popularity(users):
    all_probs = np.zeros((num_users, num_movies))

    for u in users:
        r_hat = predict_user_ratings(u)
        _, φ_u_f = compute_theme_interest(r_hat)
        all_probs[u] = request_probability(r_hat, φ_u_f)

    return all_probs.sum(axis=0) / np.sum(all_probs)

In [ ]:
def compute_rsu_global_popularity(vehicles_with_popularity):
    all_local = np.stack([v['local_popularity'] for v in vehicles_with_popularity])
    return all_local.sum(axis=0) / np.sum(all_local)

In [ ]:
with tqdm(total=VEHICLE_COUNT, desc="Vehicle Popularities") as pbar:
    for vehicle in vehicles:
        if vehicle['initial_rsu_index'] is None:
            continue

        rsu_model = rsus[vehicle['initial_rsu_index']]['model']
        vehicle_ratings_norm = ratings_matrix_norm[vehicle['users']]
        infos = user_features[vehicle['users'], :3]
        vehicle['local_popularity'] = compute_vehicle_popularity(vehicle['users'])
        pbar.update(1)


for rsu in rsus:
    rsu_vehicles = [v for v in vehicles if v['initial_rsu_index'] == rsu['id'] and 'local_popularity' in v]

    if not rsu_vehicles:
        print(f"Warning: RSU {rsu['id']} has no assigned vehicles for popularity computation.")
        rsu['global_popularity'] = np.zeros(genre_matrix.shape[0])  # or skip, or assign None
        continue

    rsu['global_popularity'] = compute_rsu_global_popularity(rsu_vehicles)

In [ ]:
# Reward weights: λ1+λ2+λ3+λ4=1 and λ1<λ2<λ3<<λ4
LAMBDA = dict(l1=0.001, l2=0.01, l3=0.39, l4=0.599)

# χ scales for reward branches (χ0..χ4)
CHI = dict(c0=1.00, c1=0.95, c2=0.90, c3=0.80, c4=0.50)

# Radio model
B = 540_000                 # 540 KHz bandwidth (Hz)
SIGMA2 = 3.98e-15           # σ^2 noise power: -114 dBm -> 3.98e-15 W

# Tx powers (W) from dBm: p_v=3 dBm, p_s=30 dBm, p_M=43 dBm
P_V = 0.002                 # 3 dBm
P_S = 1.0                   # 30 dBm
P_M = 19.95                # 43 dBm

WIRED_R_RP = 100e6          # R_{R,R'} fixed wired (bits/s)
MBS_DISTANCE = 100_000.0      # meters

# Large-scale path loss + log-normal shadowing (used by channel_gain below)
PATHLOSS_EXP = 3.0
PL_K = 1.0
SHADOW_FADING_STD_DB = 4.0  # σ (dB) for shadowing

# training
GAMMA = 0.99                # ζ 
LR    = 1e-2                # η_δ

# Cache/content sizes (bits) — the rate functions output bits/s
CONTENT_SIZE_BITS  = 5 * 1024 * 1024 * 8     # 5 MB per object, in bits
RSU_CACHE_ITEMS    = 1000
VEH_CACHE_ITEMS    = 1000
RSU_CACHE_BITS     = RSU_CACHE_ITEMS * CONTENT_SIZE_BITS
VEH_CACHE_BITS     = VEH_CACHE_ITEMS * CONTENT_SIZE_BITS

In [ ]:
import math
import numpy as np

def channel_gain(distance_m: float) -> float:
    """
    h_n(d) = path_loss(d) * shadow(d)
    We use 1/d^n with log-normal shadowing.
    """
    d = max(distance_m, 1.0)
    path_loss = PL_K / (d ** PATHLOSS_EXP)
    shadow_db = np.random.normal(0.0, SHADOW_FADING_STD_DB)
    shadow_lin = 10 ** (shadow_db / 10.0)
    return path_loss * shadow_lin

# Shannon rates for links
def _rate(power_w, d_m):
    h = channel_gain(d_m)
    return B * math.log2(1.0 + (power_w * h) / SIGMA2)  # bits/s

def rate_v2v(d_m):   return _rate(P_V, d_m)
def rate_v2r(d_m):   return _rate(P_S, d_m)
def rate_v2mbs(d_m): return _rate(P_M, d_m)

def delay_v2v(d_m):
    R = max(rate_v2v(d_m), 1e-9)
    return CONTENT_SIZE_BITS / R

def delay_v2r(d_m):
    R = max(rate_v2r(d_m), 1e-9)
    return CONTENT_SIZE_BITS / R

def delay_r2r_plus_v2r(d_m):
    # τ_vn,S' = τ_vn,S + (F_f / R_{R,R'})
    return delay_v2r(d_m) + (CONTENT_SIZE_BITS / max(WIRED_R_RP, 1.0))

def delay_mbs(d_m):
    R = max(rate_v2mbs(d_m), 1e-9)
    return CONTENT_SIZE_BITS / R

def compute_reward_from_source(source: str, delays: dict) -> float:
    """
    branch by source and apply exp(-λ τ) with χ scale.
    source ∈ {"self","v2v","rsu","neighbor_rsu","mbs"}
    delays: dict with needed τ values (seconds)
    """
    l1, l2, l3, l4 = LAMBDA['l1'], LAMBDA['l2'], LAMBDA['l3'], LAMBDA['l4']
    if source == "self":
        return CHI['c0']  # τ=0

    if source == "v2v":
        τ = delays["v2v"]
        return CHI['c1'] * math.exp(-l1 * τ)

    if source == "rsu":
        τ = delays["v2r"]
        return CHI['c2'] * math.exp(-l2 * τ)

    if source == "neighbor_rsu":
        τ_local  = delays["v2r"]
        τ_wired  = delays["r2r"]  # F/R_RR'
        return CHI['c3'] * math.exp(-(l2 * τ_local + l3 * τ_wired))

    if source == "mbs":
        τ = delays["mbs"]
        return CHI['c4'] * math.exp(-l4 * τ)

    return 0.0

In [ ]:
def reset_episode(vehicles, rsus):
    """
    Reset positions, RSU membership, and caches at episode start
    """
    for v in vehicles:
        v['position'] = v['initial_position']
        v['rsu_index'] = v['initial_rsu_index']
        v['cache'] = []                # ordered list (most-popular first)
        v['cache_set'] = set()         # fast membership
    for r in rsus:
        r['cache'] = []
        r['cache_set'] = set()

def _fill_cache_by_popularity(cache_list, cache_set, popularity_vec, capacity_items):
    top_ids = np.argsort(popularity_vec)[::-1]
    for f in top_ids:
        if len(cache_list) >= capacity_items: break
        if f not in cache_set:
            cache_list.append(int(f))
            cache_set.add(int(f))

def init_caches_from_popularity(rsus, vehicles):
    """
    After popularity prediction (Alg. 1 finished), cache top-ranked contents:
      - RSU: use rsu['global_popularity']  (O_f)
      - Vehicle: use vehicle['local_popularity']  (O_f^{v_n})
    """
    # RSUs
    for r in rsus:
        pop = r.get('global_popularity', None)
        if pop is None:  # edge: fill empty
            r['cache'], r['cache_set'] = [], set()
            continue
        r['cache'], r['cache_set'] = [], set()
        _fill_cache_by_popularity(r['cache'], r['cache_set'], pop, RSU_CACHE_ITEMS)

    # Vehicles
    for v in vehicles:
        pop = v.get('local_popularity', None)
        v['cache'], v['cache_set'] = [], set()
        if pop is None:    # default empty
            continue
        _fill_cache_by_popularity(v['cache'], v['cache_set'], pop, VEH_CACHE_ITEMS)

import copy

def snapshot_state(rsus, vehicles):
    return copy.deepcopy(rsus), copy.deepcopy(vehicles)

def restore_state(snap_rsus, snap_vehicles):
    rsus[:] = copy.deepcopy(snap_rsus)
    vehicles[:] = copy.deepcopy(snap_vehicles)

def hard_reset_episode(rsus, vehicles):
    """
    Full reset for fair runs:
    - reset vehicles to initial positions/routes
    - recompute coverage (rsu_index / prev_rsu_index)
    - (re)initialize caches from popularity
    - (re)init priorities & buffers
    """
    reset_episode(vehicles, rsus)
    update_vehicle_rsu_assignments(vehicles, rsus)
    for v in vehicles:
        v["prev_rsu_index"] = v.get("rsu_index", None)

    if any(len(r.get("cache", [])) == 0 for r in rsus):
        init_caches_from_popularity(rsus, vehicles)

    for r in rsus:
        base = r.get("global_popularity")
        r["priority"] = np.asarray(base, dtype=np.float32).copy()
        r["arrival_buffer"] = []
        if "priority_eff" in r:
            del r["priority_eff"]

In [ ]:
import os, random, numpy as np, torch

CHECKPOINT_PATH = "checkpoint/save/gat_ppo_fullstate.pt"
CHECKPOINT_PATH_LOAD = "checkpoint/load/gat_ppo_fullstate.pt"

def save_gat_checkpoint(
    update_idx,
    gat_encoder, policy, critic,
    opt_actor, opt_critic,
    rsus, vehicles,
    t_ptr, prev_ts,
    path=CHECKPOINT_PATH,
):
    """
    Save EVERYTHING needed to resume PPO + simulation from the same point:
      - models & optimizers
      - RSU/vehicle state (positions, caches, priorities, etc.)
      - timeline index t_ptr and prev_ts
      - RNG states (numpy, python.random, torch CPU/CUDA)
    """
    snap_rsus, snap_vehicles = snapshot_state(rsus, vehicles)

    state = {
        "update_idx": int(update_idx),
        "t_ptr": int(t_ptr),
        "prev_ts": int(prev_ts),

        # models
        "gat_encoder": gat_encoder.state_dict(),
        "policy": policy.state_dict(),
        "critic": critic.state_dict(),

        # optimizers
        "opt_actor": opt_actor.state_dict(),
        "opt_critic": opt_critic.state_dict(),

        # RNG states
        "rng": {
            "numpy": np.random.get_state(),
            "python": random.getstate(),
            "torch_cpu": torch.get_rng_state(),
            "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },

        # environment (simulation state)
        "env": {
            "rsus": snap_rsus,
            "vehicles": snap_vehicles,
        },
    }

    torch.save(state, path)
    print(f"[ckpt] Saved GAT/PPO checkpoint at update {update_idx} -> {path}")


def load_gat_checkpoint(
    gat_encoder, policy, critic,
    opt_actor=None, opt_critic=None,
    path=CHECKPOINT_PATH_LOAD,
    map_location=None,
):
    """
    Load a full-state checkpoint. Returns None if not found, otherwise:
      dict(update_idx, t_ptr, prev_ts, rsus, vehicles)
    and also restores RNG states.
    """
    if map_location is None:
        map_location = "cuda" if torch.cuda.is_available() else "cpu"

    if not os.path.isfile(path):
        print(f"[ckpt] No checkpoint found at {path}, starting from scratch.")
        return None

    state = torch.load(path, map_location=map_location)

    # models
    gat_encoder.load_state_dict(state["gat_encoder"])
    policy.load_state_dict(state["policy"])
    critic.load_state_dict(state["critic"])

    # optimizers
    if opt_actor is not None and "opt_actor" in state:
        opt_actor.load_state_dict(state["opt_actor"])
    if opt_critic is not None and "opt_critic" in state:
        opt_critic.load_state_dict(state["opt_critic"])

    rng = state.get("rng", {})
    if "numpy" in rng:
        np.random.set_state(rng["numpy"])
    if "python" in rng:
        random.setstate(rng["python"])
    if "torch_cpu" in rng:
        torch.set_rng_state(rng["torch_cpu"])
    if torch.cuda.is_available() and rng.get("torch_cuda") is not None:
        torch.cuda.set_rng_state_all(rng["torch_cuda"])

    env = state.get("env", {})
    rsus_snap = env.get("rsus", None)
    vehicles_snap = env.get("vehicles", None)

    update_idx = int(state.get("update_idx", 0))
    t_ptr = int(state.get("t_ptr", 0))
    prev_ts = int(state.get("prev_ts", 0))

    print(f"[ckpt] Loaded GAT/PPO checkpoint from {path} (update {update_idx})")

    return dict(
        update_idx=update_idx,
        t_ptr=t_ptr,
        prev_ts=prev_ts,
        rsus=rsus_snap,
        vehicles=vehicles_snap,
    )

In [ ]:
def rsu_neighbors_map(rsus):
    nbrs = {}
    for r in rsus:
        rid = r['id']
        nbrs[rid] = [n for n in (rid-1, rid+1) if 0 <= n < len(rsus)]
    return nbrs

def find_source_and_delays(req_vehicle, content_id, rsu, vehs_here, rsus, neighbors):
    """
    Figure out where we can get 'content_id' from (priority order per paper):
      self -> V2V (same RSU) -> RSU -> neighbor RSU (via wired) -> MBS
    Returns: source string and dict of delays to feed the reward.
    """
    # 0) local cache
    if content_id in req_vehicle['cache_set']:
        return "self", dict()

    def dist_to_rsu(v, r): return abs(v['position'] - r['position'])

    # 1) V2V inside same RSU
    for other in vehs_here:
        if other['id'] == req_vehicle['id']:
            continue
        if content_id in other['cache_set']:
            # approximate distance for V2V
            d = abs(req_vehicle['position'] - other['position'])
            return "v2v", dict(v2v=delay_v2v(d))

    # 2) current RSU
    if content_id in rsu['cache_set']:
        d = dist_to_rsu(req_vehicle, rsu)
        return "rsu", dict(v2r=delay_v2r(d))

    # 3) neighbor RSU via wired
    for nid in neighbors[rsu['id']]:
        nr = rsus[nid]
        if content_id in nr['cache_set']:
            d = dist_to_rsu(req_vehicle, rsu)     # vehicle→local RSU
            return "neighbor_rsu", dict(v2r=delay_v2r(d),
                                        r2r=CONTENT_SIZE_BITS / WIRED_R_RP)

    # 4) MBS
    return "mbs", dict(mbs=delay_mbs(MBS_DISTANCE))

def step_move_vehicles(vehicles, dt, road_length=ROAD_LENGTH):
    """
    Move each vehicle forward by v['speed'] * dt meters.
    Positions are wrapped on a circular road of length `road_length`
    so vehicles never leave the simulation domain.

    Args:
        vehicles (list[dict]): each has keys 'position' (m) and 'speed' (m/s)
        dt (float): elapsed time in seconds since the previous timestamp
        road_length (float): total road length in meters
    """
    if dt <= 0:
        return
    L = float(road_length)
    for v in vehicles:
        # advance
        new_pos = v['position'] + v['speed'] * dt
        # wrap around road
        v['position'] = new_pos % L


def step_move_vehicles_clamped(vehicles, dt, road_length=ROAD_LENGTH):
    if dt <= 0:
        return
    L = float(road_length)
    for v in vehicles:
        v['position'] = max(0.0, min(L, v['position'] + v['speed'] * dt))


from collections import defaultdict

def build_requests_by_timestamp(df_ratings, user_id_to_idx, movie_id_to_idx):
    ts_to_reqs = defaultdict(list)
    for row in df_ratings.itertuples(index=False):
        u = user_id_to_idx.get(row.UserID, None)
        f = movie_id_to_idx.get(row.MovieID, None)
        if u is not None and f is not None:
            ts_to_reqs[row.Timestamp].append((u, f))
    ts_sorted = sorted(ts_to_reqs.keys())
    return ts_sorted, ts_to_reqs

def build_user_to_vehicle_map(vehicles):
    m = {}
    for v in vehicles:
        for u in v['users']:
            m[u] = v['id']
    return m

def group_requests_for_timestamp(ts_requests, vehicles, user2veh):
    """
    Returns dict[rsu_id] -> list of (vehicle_obj, content_id)
    """
    rsu_batches = defaultdict(list)
    for (u_idx, f_idx) in ts_requests:
        vid = user2veh.get(u_idx, None)
        if vid is None:
            continue
        veh = vehicles[vid]
        rid = veh.get('rsu_index', None)
        if rid is None:
            continue
        rsu_batches[rid].append((veh, f_idx))
    return rsu_batches

def vehicles_under_rsu(rsu_id, vehicles):
    return [v for v in vehicles if v['rsu_index'] == rsu_id]

def group_requests_by_rsu(requests, user_to_vehicle, vehicles):
    """
    Group a batch of requests by the RSU that will serve them.

    Args:
        requests: list of (user_id, item_id) for this timestamp.
        user_to_vehicle: dict {user_id: vehicle_obj}.
        vehicles: list of all vehicle objects (each must have 'rsu_index').

    Returns:
        dict {rsu_id: list of (vehicle_obj, item_id)}.
    """
    rsu_batches = {}
    for u, fid in requests:
        v = user_to_vehicle.get(u, None)
        if v is None:
            continue
        v = vehicles[v]
        rid = v.get("rsu_index", None)
        if rid is None:
            continue
        rsu_batches.setdefault(rid, []).append((v, fid))
    return rsu_batches

In [ ]:
import numpy as np
import torch

class CacheDiversityController:
    """
    Computes duplication-aware effective priorities per RSU, to reduce redundant caching
    across neighboring RSUs while preserving local demand.
    """
    def __init__(self, alpha_dup=0.5, beta_local=0.15, min_neighbors=1,
                 consider_latency=False, max_r2r_latency_ms=8.0):
        """
        alpha_dup:   how strongly to penalize items cached by neighbors (0..1)
        beta_local:  boost for items locally important (arrival buffer / recent)
        min_neighbors: normalize presence by max(1, #neighbors) to avoid div by 0
        consider_latency: if True, ignore neighbors whose r2r latency is too high
        max_r2r_latency_ms: max r2r latency to consider a neighbor "reachable"
        """
        self.alpha_dup = float(alpha_dup)
        self.beta_local = float(beta_local)
        self.min_neighbors = int(min_neighbors)
        self.consider_latency = bool(consider_latency)
        self.max_r2r_latency_ms = float(max_r2r_latency_ms)

    def _neighbor_ids(self, rsu_id, neighbors, r2r_latency_map=None):
        nbrs = list(neighbors[rsu_id])
        if not self.consider_latency or not r2r_latency_map:
            return nbrs
        ok = []
        for n in nbrs:
            key = (rsu_id, n) if (rsu_id, n) in r2r_latency_map else (n, rsu_id)
            if key in r2r_latency_map and r2r_latency_map[key] <= self.max_r2r_latency_ms:
                ok.append(n)
        return ok

    def update_effective_priorities(self, rsus, neighbors, r2r_latency_map=None, local_recent_requests=None):
        """
        Writes rsu['priority_eff'] for each RSU.

        local_recent_requests: optional dict {rsu_id: set([item_ids])} to boost recent local demand
        """
        F = len(rsus[0].get("priority", rsus[0]["global_popularity"]))
        cache_sets = {r["id"]: set(r.get("cache_set", set(r.get("cache", [])))) for r in rsus}

        for r in rsus:
            rid = r["id"]
            pr = np.asarray(r.get("priority", r.get("global_popularity")), dtype=np.float32)
            nbr_ids = self._neighbor_ids(rid, neighbors, r2r_latency_map)
            n_nbrs = max(self.min_neighbors, len(nbr_ids))

            presence = np.zeros(F, dtype=np.float32)
            if n_nbrs > 0:
                for nid in nbr_ids:
                    for f in cache_sets[nid]:
                        presence[f] += 1.0
                presence /= float(n_nbrs)

            local_boost = np.zeros(F, dtype=np.float32)
            for f in r.get("arrival_buffer", []):
                local_boost[f] = 1.0
            if local_recent_requests and rid in local_recent_requests:
                for f in local_recent_requests[rid]:
                    local_boost[f] = 1.0

            p_eff = pr * (1.0 - self.alpha_dup * presence) + self.beta_local * local_boost

            p_eff = np.clip(p_eff, 0.0, None)
            r["priority_eff"] = p_eff

div_controller = CacheDiversityController(
    alpha_dup=0.5,
    beta_local=0.15,
    min_neighbors=1,
    consider_latency=False
)

In [ ]:
# === GAT + PPO setup (paste after step 5) =====================================
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.distributions import Categorical
from collections import defaultdict
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(42)

TOPK_S = 64            # how many top priority scores to expose as RSU features
ARRIVAL_TOPK = 12      # how many items to consider in arrival buffer
MAX_PERSISTENT_SWAPS = 4
ROLL_STEPS = 256       # PPO rollout length
PPO_UPDATES = 400      # PPO update count
MINIBATCH_EPOCHS = 4
MINIBATCH_SPLITS = 4
GAMMA, LAMBDA_GAE = 0.99, 0.95
LR_ACTOR = 3e-4
LR_CRITIC = 3e-4

len_catalog = int(genre_matrix.shape[0])
if "cache_capacity" in rsus[0]:
    RSU_CACHE_ITEMS = int(rsus[0]["cache_capacity"])
elif "C" in rsus[0]:
    RSU_CACHE_ITEMS = int(rsus[0]["C"])
else:
    RSU_CACHE_ITEMS = max(len(rsus[0].get("cache", [])), 64)

for r in rsus:
    base = r.get("global_popularity", None)
    if base is None:
        base = np.zeros((len_catalog,), np.float32)
    r["priority"] = np.asarray(base, dtype=np.float32).copy()
    r.setdefault("arrival_buffer", [])

In [ ]:
_item_key = nn.Sequential(
    nn.Linear(genre_matrix.shape[1], 128, bias=False),
    nn.ReLU(),
    nn.Linear(128, 128, bias=False),
).to(device).eval()

with torch.no_grad():
    ITEM_EMB = _item_key(torch.from_numpy(genre_matrix).float().to(device)).cpu()  # [F, 128]

class ArrivalAttention(nn.Module):
    def __init__(self, d_veh, d_item, d_attn=128, beta=0.6, lam=0.5, topk_arrival=12):
        super().__init__()
        self.Wq = nn.Linear(d_veh, d_attn, bias=False)
        self.Wk = nn.Linear(d_item, d_attn, bias=False)
        self.beta = float(beta)
        self.lam  = float(lam)
        self.topk_arrival = int(topk_arrival)

    @torch.no_grad()
    def boost(self, rsu, vehicle, item_emb, dwell_time, neighbors, all_rsus):
        uidx = vehicle.get("users", [])
        if not uidx:
            return
        v_emb = torch.from_numpy(encoded_users[uidx]).float().mean(dim=0).to(device)  # [d_veh]
        q = self.Wq(v_emb)                         # [d_attn]
        K = self.Wk(item_emb.to(device))           # [F, d_attn]
        scores = K @ q                             # [F]
        pi = torch.softmax(scores, dim=0).cpu()    # [F]

        F_total = item_emb.size(0)
        red = torch.zeros(F_total)
        nbr_ids = neighbors[rsu["id"]]
        for nr in nbr_ids:
            for f in all_rsus[nr].get("cache", []):
                red[f] += 1.0
        if len(nbr_ids) > 0:
            red = red / float(len(nbr_ids))
        red = red.clamp_(0, 1)

        speed = max(1e-3, vehicle.get("speed", 10.0))
        rsu_radius = float(rsu.get("radius", 150.0))
        T0 = max(1.0, 2 * rsu_radius / speed)
        dwell_factor = min(1.0, float(dwell_time) / T0)

        pr = torch.from_numpy(rsu.get("priority")).float()
        boost = self.beta * pi * dwell_factor * (1.0 - red)  # [F]
        pr_new = (1.0 - self.lam) * pr + self.lam * boost
        rsu["priority"] = pr_new.numpy()

        order = torch.argsort(pr_new, descending=True).tolist()
        buf = []
        cached = rsu.get("cache_set", set())
        for f in order:
            if f not in cached:
                buf.append(int(f))
            if len(buf) >= self.topk_arrival:
                break
        rsu["arrival_buffer"] = buf

arrival_attn = ArrivalAttention(
    d_veh=encoded_users.shape[1], d_item=ITEM_EMB.shape[1],
    d_attn=128, beta=0.6, lam=0.5, topk_arrival=ARRIVAL_TOPK
).to(device).eval()

def handle_vehicle_arrivals(rsus, vehicles, arrival_attn, item_emb, neighbors):
    for v in vehicles:
        prev = v.get("prev_rsu_index", None)
        cur  = v.get("rsu_index", None)
        if cur is not None and cur != prev:
            dwell = calculate_staying_time(v, rsus[cur])
            arrival_attn.boost(
                rsu=rsus[cur], vehicle=v, item_emb=item_emb,
                dwell_time=dwell, neighbors=neighbors, all_rsus=rsus
            )
    for v in vehicles:
        v["prev_rsu_index"] = v.get("rsu_index", None)

In [ ]:
# === Controller-aware patches ==================================================

def build_graph_inputs(rsus, vehicles, encoded_users, topk_s=TOPK_S):
    """
    Use rsu['priority_eff'] when available (else 'priority' -> 'global_popularity').
    """
    import numpy as np
    import torch

    len_catalog = len(rsus[0].get("priority", rsus[0].get("global_popularity")))
    R, V = len(rsus), len(vehicles)

    x_r = []
    for r in rsus:
        pr = r.get("priority_eff", r.get("priority", r.get("global_popularity", np.zeros((len_catalog,), np.float32))))
        pr = np.asarray(pr, dtype=np.float32)
        order = np.argsort(pr)[::-1][:topk_s]
        vec = np.zeros((topk_s,), np.float32)
        vec[:len(order)] = pr[order]
        x_r.append(torch.from_numpy(vec))
    x_r = torch.stack(x_r, dim=0).float()  # [R, topk_s]

    d_u = encoded_users.shape[1]
    def vehicle_embed(v):
        uidx = v.get("users", [])
        if not uidx: return torch.zeros(d_u)
        return torch.from_numpy(encoded_users[uidx]).float().mean(dim=0)
    x_v = torch.stack([vehicle_embed(v) for v in vehicles], dim=0).float()  # [V, d_u]

    adj_rv = [[] for _ in range(R)]
    for vid, v in enumerate(vehicles):
        rid = v.get("rsu_index", None)
        if rid is not None:
            adj_rv[rid].append(vid)

    nbrs = rsu_neighbors_map(rsus)  # you already have this
    adj_rr = [nbrs[r["id"]] for r in rsus]

    return x_r, x_v, adj_rv, adj_rr


def apply_persistent_swaps(rsu, n_swaps, cap_items):
    """
    Use effective priorities for decisions (priority_eff -> priority -> global_popularity).
    """
    import numpy as np

    if n_swaps <= 0: return
    pr = np.asarray(rsu.get("priority_eff", rsu.get("priority", rsu.get("global_popularity"))), dtype=np.float32)

    victims = []
    if rsu["cache"]:
        cur_scores = np.array([pr[f] for f in rsu["cache"]])
        victims = [rsu["cache"][i] for i in np.argsort(cur_scores)[:n_swaps]]

    cand = [int(i) for i in np.argsort(pr)[::-1] if i not in rsu["cache_set"]]
    added = 0
    for f_new in cand:
        if victims:
            f_old = victims.pop(0)
            if f_old in rsu["cache_set"]:
                rsu["cache_set"].remove(f_old)
                rsu["cache"].remove(f_old)
        if len(rsu["cache"]) < cap_items:
            rsu["cache"].append(f_new)
            rsu["cache_set"].add(f_new)
            added += 1
        if added >= n_swaps:
            break

    s_pop = rsu.get("global_popularity", None)
    if s_pop is not None:
        rsu["cache"].sort(key=lambda fid: s_pop[fid], reverse=True)


class SimpleGATLayer(nn.Module):
    def __init__(self, q_dim, k_dim, out_dim, n_heads=4):
        super().__init__()
        self.nh = n_heads
        self.Wq = nn.Linear(q_dim, out_dim * n_heads, bias=False)
        self.Wk = nn.Linear(k_dim, out_dim * n_heads, bias=False)
        self.Wv = nn.Linear(k_dim, out_dim * n_heads, bias=False)

    def forward(self, Q, K, nbrs_idx):
        """
        Q: [R, q_dim]  (queries per RSU)
        K: [N, k_dim]  (keys/values: vehicles or RSUs)
        nbrs_idx: list of lists; for RSU r, indices into K that are its neighbors
        """
        R = Q.size(0)
        q = self.Wq(Q).view(R, self.nh, -1)    # [R, H, d]
        k_all = self.Wk(K)                      # [N, H*d]
        v_all = self.Wv(K)                      # [N, H*d]
        d = q.size(-1)

        outs = []
        for r in range(R):
            idx = nbrs_idx[r]
            if not idx:
                outs.append(torch.zeros(self.nh, d, device=Q.device))
                continue

            k = k_all[idx].view(len(idx), self.nh, d)    # [Nr, H, d]
            v = v_all[idx].view(len(idx), self.nh, d)    # [Nr, H, d]

            k_hnd = k.transpose(0, 1)  # [H, Nr, d]
            v_hnd = v.transpose(0, 1)  # [H, Nr, d]

            q_h1d = q[r].unsqueeze(1)  # [H, 1, d]

            scores = torch.sum(q_h1d * k_hnd, dim=-1)

            alpha = torch.softmax(scores, dim=-1)  # [H, Nr]

            ctx_hnd = (alpha.unsqueeze(-1) * v_hnd).sum(dim=1)  # [H, d]

            outs.append(ctx_hnd)

        out = torch.stack(outs, dim=0).reshape(R, -1)  # [R, H*d]
        return out


class FastGATLayer(nn.Module):
    def __init__(self, q_dim, k_dim, out_dim, n_heads=4):
        super().__init__()
        self.nh = n_heads
        self.Wq = nn.Linear(q_dim, out_dim * n_heads, bias=False)
        self.Wk = nn.Linear(k_dim, out_dim * n_heads, bias=False)
        self.Wv = nn.Linear(k_dim, out_dim * n_heads, bias=False)

    def forward(self, Q, K, nbrs_idx):
        """
        Q: [R, q_dim], K: [N, k_dim], nbrs_idx: list of lists (len R)
        """
        device = Q.device
        R = Q.size(0)
        H = self.nh
        d = (self.Wq.out_features // H)

        lengths = torch.tensor([len(ix) for ix in nbrs_idx], device=device)
        M = int(lengths.max().item()) if R > 0 else 0
        if M == 0:
            # no neighbors anywhere
            q = self.Wq(Q).view(R, H, d)
            return torch.zeros(R, H * d, device=device)

        idx_pad = torch.full((R, M), fill_value=-1, device=device, dtype=torch.long)
        for r, ix in enumerate(nbrs_idx):
            if len(ix):
                idx_pad[r, :len(ix)] = torch.tensor(ix, device=device)

        mask = (idx_pad != -1)                                 # [R, M]
        idx_safe = idx_pad.clamp(min=0)
        K_all = self.Wk(K)                                     # [N, H*d]
        V_all = self.Wv(K)                                     # [N, H*d]
        K_rm = K_all.index_select(0, idx_safe.view(-1)).view(R, M, H*d)
        V_rm = V_all.index_select(0, idx_safe.view(-1)).view(R, M, H*d)
        K_rm = K_rm.view(R, M, H, d).permute(0,2,1,3)          # [R, H, M, d]
        V_rm = V_rm.view(R, M, H, d).permute(0,2,1,3)          # [R, H, M, d]

        # Project Q and compute scores in batch
        Q_r = self.Wq(Q).view(R, H, d)                         # [R, H, d]
        Q_r = Q_r.unsqueeze(2)                                 # [R, H, 1, d]
        scores = (Q_r * K_rm).sum(-1)                          # [R, H, M]

        # Masked softmax over neighbors
        mask_h = mask.unsqueeze(1)                             # [R, 1, M]
        scores = scores.masked_fill(~mask_h, -1e9)
        alpha = torch.softmax(scores, dim=-1)                  # [R, H, M]

        # Context: weighted sum over V
        ctx = (alpha.unsqueeze(-1) * V_rm).sum(dim=2)          # [R, H, d]
        return ctx.reshape(R, H * d)


class RSUGATEncoder(nn.Module):
    def __init__(self, fr_rsu, fv_veh, hidden=128, heads=4, out_dim=256):
        super().__init__()
        self.veh_attn = FastGATLayer(q_dim=fr_rsu, k_dim=fv_veh, out_dim=hidden, n_heads=heads)
        self.rsu_attn = FastGATLayer(q_dim=fr_rsu, k_dim=fr_rsu, out_dim=hidden, n_heads=heads)
        self.proj = nn.Sequential(
            nn.Linear(fr_rsu + hidden*heads + hidden*heads, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
    def forward(self, x_r, x_v, adj_rv, adj_rr):
        x_r = x_r.to(device); x_v = x_v.to(device)
        veh_ctx = self.veh_attn(x_r, x_v, adj_rv)  # batched
        rsu_ctx = self.rsu_attn(x_r, x_r, adj_rr)  # batched
        return self.proj(torch.cat([x_r, veh_ctx, rsu_ctx], dim=1))


gat_encoder = RSUGATEncoder(fr_rsu=TOPK_S, fv_veh=encoded_users.shape[1], hidden=128, heads=4, out_dim=256).to(device)

In [ ]:
class GATPolicy(nn.Module):
    def __init__(self, z_dim=256, max_swaps=MAX_PERSISTENT_SWAPS):
        super().__init__()
        self.max_swaps = max_swaps
        self.actor = nn.Sequential(
            nn.Linear(z_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, max_swaps + 1)  # categorical logits for {0..M}
        )

    @torch.no_grad()
    def act(self, z_r, rsu):
        logits = self.actor(z_r.unsqueeze(0)).squeeze(0)
        n_swaps = int(torch.argmax(logits).item())
        ephemeral = rsu.get("arrival_buffer", [])[:min(len(rsu.get("arrival_buffer", [])), 8)]
        return {"n_swaps": n_swaps, "ephemeral_ids": ephemeral}

policy = GATPolicy(z_dim=256, max_swaps=MAX_PERSISTENT_SWAPS).to(device)

def apply_ephemeral_inserts(rsu, ephemeral_ids, cap_items):
    for f in ephemeral_ids:
        if f in rsu["cache_set"]: continue
        if len(rsu["cache"]) < cap_items:
            rsu["cache"].append(f)
            rsu["cache_set"].add(f)
    s_pop = rsu.get("global_popularity", None)
    if s_pop is not None:
        rsu["cache"].sort(key=lambda fid: s_pop[fid], reverse=True)


In [ ]:
def compute_rsu_rewards(request_batch, rsu, vehs_here, rsus, neighbors):
    if not request_batch:
        return 0.0, 0, 0.0
    hits, delays = 0, []
    for (veh, fid) in request_batch:
        src, dly = find_source_and_delays(veh, fid, rsu, vehs_here, rsus, neighbors)
        if   src == "self":          tau = 0.0;                    hit = True
        elif src == "v2v":           tau = dly["v2v"];              hit = True
        elif src == "rsu":           tau = dly["v2r"];              hit = True
        elif src == "neighbor_rsu":  tau = dly["v2r"] + dly["r2r"]; hit = True
        else:                        tau = dly["mbs"];              hit = False
        hits += (1 if hit else 0)
        delays.append(tau)
    hit_ratio = hits / max(1, len(request_batch))
    avg_delay = (sum(delays)/len(delays)) if delays else 0.0
    reward = 1.0 * hit_ratio - 0.1 * avg_delay
    return reward, hits, avg_delay

class CentralCritic(nn.Module):
    def __init__(self, z_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, z_r):  # [R, z_dim]
        return self.net(z_r).squeeze(-1)  # [R]

class RolloutBuffer:
    def __init__(self):
        self.data = defaultdict(list)
    def add(self, **kv):
        for k, v in kv.items():
            self.data[k].append(v)
    def stack(self, device):
        out = {}
        for k, v in self.data.items():
            if isinstance(v[0], torch.Tensor):
                out[k] = torch.stack(v).to(device)
            else:
                out[k] = torch.tensor(v, dtype=torch.float32, device=device)
        return out
    def clear(self):
        self.data.clear()

critic = CentralCritic(z_dim=256).to(device)

In [ ]:
import time
import csv
from collections import Counter
import numpy as np


def compute_neighbor_cache_overlap(rsus, neighbors):
    """
    Average Jaccard overlap of each RSU cache with its neighbors' caches.
    Returns: overlap_avg, overlap_std
    """
    overlaps = []
    for r in rsus:
        rid = r["id"]
        c_i = set(r.get("cache_set", set(r.get("cache", []))))
        if len(c_i) == 0:
            continue
        for nid in neighbors[rid]:
            c_j = set(rsus[nid].get("cache_set", set(rsus[nid].get("cache", []))))
            if len(c_j) == 0:
                continue
            inter = len(c_i & c_j)
            union = len(c_i | c_j)
            overlaps.append(inter / max(1, union))
    if not overlaps:
        return 0.0, 0.0
    return float(np.mean(overlaps)), float(np.std(overlaps))


from collections import Counter
import numpy as np
import torch

def evaluate_short_window(
    rsus, vehicles,
    ts_list, ts_to_reqs,
    user_id_to_idx, movie_id_to_idx,
    gat_encoder, policy,
    arrival_attn, ITEM_EMB, encoded_users,
    steps=200, topk_s=64, device="cpu",
    restore_env=True,
    return_delay_percentiles=True,
    percentiles=(50, 95, 99),
):
    """
    Short evaluation for training-time validation:
    - hit_ratio, avg_delay, delay percentiles (optional), source_distribution
    - runs only `steps` timestamps
    - optionally restores environment state afterwards (recommended)
    """
    if restore_env:
        snap_rsus, snap_vehicles = snapshot_state(rsus, vehicles)

    neighbors = rsu_neighbors_map(rsus)
    for v in vehicles:
        v["prev_rsu_index"] = v.get("rsu_index", None)

    total_reqs, total_hits = 0, 0
    delays = []
    src_counter = Counter()

    prev_ts = ts_list[0]
    steps = min(int(steps), len(ts_list))

    gat_encoder.eval()
    policy.eval()

    with torch.inference_mode():
        for k in range(steps):
            t = ts_list[k]
            dt = max(0, t - prev_ts); prev_ts = t
            if dt > 0:
                step_move_vehicles(vehicles, dt)
                update_vehicle_rsu_assignments(vehicles, rsus)
                handle_vehicle_arrivals(rsus, vehicles, arrival_attn, ITEM_EMB, neighbors)

            ts_requests = ts_to_reqs[t]
            tmp_batches = group_requests_by_rsu(ts_requests, build_user_to_vehicle_map(vehicles), vehicles)
            recent_local = {rid: {fid for (_, fid) in batch} for rid, batch in tmp_batches.items()}
            div_controller.update_effective_priorities(rsus, neighbors, local_recent_requests=recent_local)

            x_r, x_v, adj_rv, adj_rr = build_graph_inputs(rsus, vehicles, encoded_users, topk_s=topk_s)
            z_r = gat_encoder(x_r, x_v, adj_rv, adj_rr)

            for rid, batch in tmp_batches.items():
                rsu = rsus[rid]
                vehs_here = vehicles_under_rsu(rid, vehicles)
                if not vehs_here:
                    continue

                act = policy.act(z_r[rid], rsu)
                apply_ephemeral_inserts(rsu, act["ephemeral_ids"], RSU_CACHE_ITEMS)
                apply_persistent_swaps(rsu, act["n_swaps"], RSU_CACHE_ITEMS)

                for (veh, fid) in batch:
                    src, dly = find_source_and_delays(veh, fid, rsu, vehs_here, rsus, neighbors)

                    if src == "self":
                        tau = 0.0; hit = True
                    elif src == "v2v":
                        tau = dly["v2v"]; hit = True
                    elif src == "rsu":
                        tau = dly["v2r"]; hit = True
                    elif src == "neighbor_rsu":
                        tau = dly["v2r"] + dly["r2r"]; hit = True
                    else:
                        tau = dly["mbs"]; hit = False

                    total_reqs += 1
                    total_hits += int(hit)
                    delays.append(float(tau))
                    src_counter[src] += 1

    out = {
        "hit_ratio": total_hits / max(1, total_reqs),
        "avg_delay": float(np.mean(delays)) if delays else 0.0,
        "requests": int(total_reqs),
        "src_self": int(src_counter["self"]),
        "src_v2v": int(src_counter["v2v"]),
        "src_rsu": int(src_counter["rsu"]),
        "src_neighbor_rsu": int(src_counter["neighbor_rsu"]),
        "src_mbs": int(src_counter["mbs"]),
    }

    if return_delay_percentiles:
        if delays:
            for p in percentiles:
                out[f"p{p}_delay"] = float(np.percentile(delays, p))
        else:
            for p in percentiles:
                out[f"p{p}_delay"] = 0.0

    if restore_env:
        restore_state(snap_rsus, snap_vehicles)

    return out





In [ ]:
import csv, time
import numpy as np
from tqdm.notebook import tqdm
from torch.distributions import Categorical

import torch
import torch.nn as nn
import torch.nn.functional as F

def train_gat_ppo(
    rsus, vehicles,
    ratings_dataset, user_id_to_idx, movie_id_to_idx,
    gat_encoder, policy, critic, arrival_attn, ITEM_EMB, encoded_users,
    steps_per_update=256, minibatch_epochs=4, minibatch_splits=4, total_updates=400,
    gamma=0.99, lam=0.95, lr_actor=3e-4, lr_critic=3e-4, topk_s=64, log_every=10,
    train_fraction=1.0, time_stride=1, skip_prob=0.0,
    show_step_bar=True, show_minibatch_bar=True,
    resume=True,
    checkpoint_path=None,
    checkpoint_path_load=None,
    checkpoint_freq=1,
    train_gat_end2end=False,
    clip_eps=0.2,
    ent_coef=0.01,
):
    gat_encoder.to(device)
    policy.to(device)
    critic.to(device)

    actor_params = list(policy.parameters()) + (list(gat_encoder.parameters()) if train_gat_end2end else [])
    opt_actor  = torch.optim.Adam(actor_params, lr=lr_actor)
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr_critic)

    ts_list, ts_to_reqs = build_requests_by_timestamp(ratings_dataset, user_id_to_idx, movie_id_to_idx)
    assert len(ts_list) > 0, "Empty dataset for training."

    if train_fraction < 1.0:
        n_keep = max(1, int(len(ts_list) * train_fraction))
        ts_list = ts_list[:n_keep]
    if time_stride > 1:
        ts_list = ts_list[::time_stride]

    # timeline state
    t_ptr = 0
    prev_ts = ts_list[0]

    # --- resume ---
    start_update = 1
    if resume and checkpoint_path_load is not None:
        ck = load_gat_checkpoint(
            gat_encoder, policy, critic,
            opt_actor=opt_actor, opt_critic=opt_critic,
            path=checkpoint_path_load, map_location=device
        )
        if ck is not None:
            start_update = ck["update_idx"] + 1
            if ck.get("rsus") is not None:     rsus[:] = ck["rsus"]
            if ck.get("vehicles") is not None: vehicles[:] = ck["vehicles"]
            t_ptr = ck.get("t_ptr", 0)
            prev_ts = ck.get("prev_ts", ts_list[0])

    # --- logging ---
    log_path = "results/train_logs.csv"
    log_fields = [
        "update",
        "rollout_reward_mean", "rollout_reward_std",
        "entropy_mean",
        "value_loss_last", "policy_loss_last",
        "approx_kl", "clipfrac", "explained_var",
        "action_mean", "action_std",
        "sec_per_update",
        "val_hit_ratio", "val_avg_delay", "val_p95_delay", "val_p99_delay", "val_requests",
        "val_src_self", "val_src_v2v", "val_src_rsu", "val_src_neighbor_rsu", "val_src_mbs",
        "cache_overlap_avg", "cache_overlap_std",
    ]
    if start_update == 1:
        with open(log_path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=log_fields).writeheader()

    def encode_graph_no_grad():
        x_r, x_v, adj_rv, adj_rr = build_graph_inputs(rsus, vehicles, encoded_users, topk_s=topk_s)
        with torch.no_grad():
            z_r = gat_encoder(x_r, x_v, adj_rv, adj_rr)
        return z_r

    def encode_graph_with_grad():
        x_r, x_v, adj_rv, adj_rr = build_graph_inputs(rsus, vehicles, encoded_users, topk_s=topk_s)
        z_r = gat_encoder(x_r, x_v, adj_rv, adj_rr)
        return z_r

    for v in vehicles:
        v["prev_rsu_index"] = v.get("rsu_index", None)
    neighbors = rsu_neighbors_map(rsus)

    buf = RolloutBuffer()

    outer_bar = tqdm(range(start_update, total_updates+1), desc="Training PPO", unit="update",
                     position=0, leave=True, dynamic_ncols=True, mininterval=0.1)

    for upd in outer_bar:
        t_update_start = time.time()

        reward_steps = []
        entropy_steps = []
        action_means = []
        action_stds = []

        value_loss_last = None
        policy_loss_last = None

        approx_kl_list = []
        clipfrac_list = []

        buf.clear()

        # rollout mode
        gat_encoder.eval()
        policy.train()
        critic.train()

        step_iter = range(steps_per_update)
        if show_step_bar:
            step_iter = tqdm(step_iter, desc=f"Update {upd} steps", unit="step",
                             position=1, leave=False, dynamic_ncols=True, mininterval=0.05)

        # -------- rollout --------
        for _ in step_iter:
            if t_ptr >= len(ts_list):
                t_ptr = 0
                prev_ts = ts_list[0]

            t = ts_list[t_ptr]
            if skip_prob > 0.0 and np.random.rand() < skip_prob:
                t_ptr += 1
                continue

            dt = max(0, t - prev_ts); prev_ts = t
            t_ptr += 1

            if dt > 0:
                step_move_vehicles(vehicles, dt)
                update_vehicle_rsu_assignments(vehicles, rsus)
                handle_vehicle_arrivals(rsus, vehicles, arrival_attn, ITEM_EMB, neighbors)

            ts_requests = ts_to_reqs[t]
            tmp_batches = group_requests_by_rsu(ts_requests, build_user_to_vehicle_map(vehicles), vehicles)
            recent_local = {rid: {fid for (_, fid) in batch} for rid, batch in tmp_batches.items()}
            div_controller.update_effective_priorities(rsus, neighbors, local_recent_requests=recent_local)

            z_r = encode_graph_no_grad().detach()  # frozen snapshot for rollout

            logits_all = policy.actor(z_r)
            dists = Categorical(logits=logits_all)
            a_swaps = dists.sample()
            logp = dists.log_prob(a_swaps)
            values = critic(z_r)

            entropy_step = dists.entropy().mean().item()

            rewards = torch.zeros_like(values)
            for rid, batch in tmp_batches.items():
                rsu = rsus[rid]
                vehs_here = vehicles_under_rsu(rid, vehicles)
                apply_ephemeral_inserts(rsu, rsu.get("arrival_buffer", [])[:8], RSU_CACHE_ITEMS)
                apply_persistent_swaps(rsu, int(a_swaps[rid].item()), RSU_CACHE_ITEMS)
                R, _, _ = compute_rsu_rewards(batch, rsu, vehs_here, rsus, neighbors)
                rewards[rid] = R

            buf.add(values=values.detach(), logp=logp.detach(), actions=a_swaps.detach(), rewards=rewards.detach())
            reward_steps.append(rewards.mean().item())
            entropy_steps.append(entropy_step)
            action_means.append(a_swaps.float().mean().item())
            action_stds.append(a_swaps.float().std(unbiased=False).item())

        # -------- PPO update --------
        data = buf.stack(device)
        with torch.no_grad():
            v_last = values.detach()

        T = data["rewards"].shape[0]
        values_b   = data["values"]     # [T, R]
        rewards_b  = data["rewards"]    # [T, R]
        logp_old_b = data["logp"]       # [T, R]
        actions_b  = data["actions"]    # [T, R]

        advantages = torch.zeros_like(rewards_b)
        returns    = torch.zeros_like(rewards_b)
        gae = torch.zeros(values_b.shape[1], device=device)

        for ti in reversed(range(T)):
            v_next = values_b[ti+1] if ti < T-1 else v_last
            delta  = rewards_b[ti] + gamma * v_next - values_b[ti]
            gae    = delta + gamma * lam * gae
            advantages[ti] = gae
            returns[ti]    = advantages[ti] + values_b[ti]

        adv = (advantages - advantages.mean()) / advantages.std().clamp_min(1e-6)

        B = T * values_b.shape[1]
        def flat2(x): return x.reshape(B).contiguous()

        actions_f  = flat2(actions_b).long().detach()
        logp_old_f = flat2(logp_old_b).detach()
        adv_f      = flat2(adv).detach()
        ret_f      = flat2(returns).detach()

        idx = torch.randperm(B, device=device)
        mb_size = max(1, B // minibatch_splits)

        # training mode for update
        if train_gat_end2end:
            gat_encoder.train()
        else:
            gat_encoder.eval()
        policy.train()
        critic.train()

        if train_gat_end2end:
            z_base = encode_graph_with_grad()
        else:
            z_base = z_r.detach()

        for ep in range(minibatch_epochs):
            mb_iter = range(minibatch_splits)
            if show_minibatch_bar:
                mb_iter = tqdm(mb_iter, desc=f"Upd {upd} Ep {ep+1}", unit="mb",
                               position=2, leave=False, dynamic_ncols=True, mininterval=0.05)

            for j in mb_iter:
                sel = idx[j*mb_size:(j+1)*mb_size]

                mb_actions  = actions_f.index_select(0, sel)
                mb_adv      = adv_f.index_select(0, sel)
                mb_ret      = ret_f.index_select(0, sel)
                mb_logp_old = logp_old_f.index_select(0, sel)

                # ---- critic ----
                values_now  = critic(z_base)  # [R]
                values_flat = values_now.unsqueeze(0).repeat(T, 1).reshape(B)  # ✅ aligned
                mb_values   = values_flat.index_select(0, sel)
                value_loss  = F.mse_loss(mb_values, mb_ret)

                opt_critic.zero_grad(set_to_none=True)
                value_loss.backward()
                nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
                opt_critic.step()

                # ---- actor (+ optionally GAT) ----
                logits_all  = policy.actor(z_base)                              # [R, A]
                logits_flat = logits_all.unsqueeze(0).repeat(T, 1, 1).reshape(B, -1)  # ✅ aligned
                mb_logits   = logits_flat.index_select(0, sel)

                dist     = Categorical(logits=mb_logits)
                mb_logp  = dist.log_prob(mb_actions)
                entropy  = dist.entropy().mean()

                ratio = torch.exp(mb_logp - mb_logp_old)
                surr1 = ratio * mb_adv
                surr2 = torch.clamp(ratio, 1.0-clip_eps, 1.0+clip_eps) * mb_adv
                policy_loss = -torch.min(surr1, surr2).mean() - ent_coef * entropy

                # PPO diagnostics
                approx_kl = (mb_logp_old - mb_logp).mean().item()
                clipfrac  = (torch.abs(ratio - 1.0) > clip_eps).float().mean().item()
                approx_kl_list.append(approx_kl)
                clipfrac_list.append(clipfrac)

                value_loss_last = float(value_loss.item())
                policy_loss_last = float(policy_loss.item())

                opt_actor.zero_grad(set_to_none=True)
                policy_loss.backward()
                nn.utils.clip_grad_norm_(actor_params, 1.0)
                opt_actor.step()

        # explained variance (critic quality)
        with torch.no_grad():
            v_pred = critic(z_base).unsqueeze(0).repeat(T, 1).reshape(B)
            y = ret_f
            explained_var = (1.0 - (y - v_pred).var() / y.var().clamp_min(1e-8)).item()

        sec_per_update = time.time() - t_update_start
        overlap_avg, overlap_std = compute_neighbor_cache_overlap(rsus, neighbors)

        # validation
        val = {"hit_ratio": np.nan, "avg_delay": np.nan, "p95_delay": np.nan, "p99_delay": np.nan, "requests": 0,
               "src_self": 0, "src_v2v": 0, "src_rsu": 0, "src_neighbor_rsu": 0, "src_mbs": 0}
        if log_every and (upd % log_every == 0):
            val = evaluate_short_window(
                rsus, vehicles,
                ts_list=ts_list, ts_to_reqs=ts_to_reqs,
                user_id_to_idx=user_id_to_idx, movie_id_to_idx=movie_id_to_idx,
                gat_encoder=gat_encoder, policy=policy,
                arrival_attn=arrival_attn, ITEM_EMB=ITEM_EMB, encoded_users=encoded_users,
                steps=200, topk_s=topk_s, device=device,
                restore_env=True,
                return_delay_percentiles=True,  # implement in your eval_short_window
            )

        row = {
            "update": upd,
            "rollout_reward_mean": float(np.mean(reward_steps)) if reward_steps else 0.0,
            "rollout_reward_std": float(np.std(reward_steps)) if reward_steps else 0.0,
            "entropy_mean": float(np.mean(entropy_steps)) if entropy_steps else 0.0,
            "value_loss_last": value_loss_last if value_loss_last is not None else np.nan,
            "policy_loss_last": policy_loss_last if policy_loss_last is not None else np.nan,
            "approx_kl": float(np.mean(approx_kl_list)) if approx_kl_list else np.nan,
            "clipfrac": float(np.mean(clipfrac_list)) if clipfrac_list else np.nan,
            "explained_var": float(explained_var),
            "action_mean": float(np.mean(action_means)) if action_means else np.nan,
            "action_std": float(np.mean(action_stds)) if action_stds else np.nan,
            "sec_per_update": float(sec_per_update),
            "val_hit_ratio": float(val["hit_ratio"]) if not np.isnan(val["hit_ratio"]) else np.nan,
            "val_avg_delay": float(val["avg_delay"]) if not np.isnan(val["avg_delay"]) else np.nan,
            "val_p95_delay": float(val.get("p95_delay", np.nan)),
            "val_p99_delay": float(val.get("p99_delay", np.nan)),
            "val_requests": int(val["requests"]),
            "val_src_self": int(val["src_self"]),
            "val_src_v2v": int(val["src_v2v"]),
            "val_src_rsu": int(val["src_rsu"]),
            "val_src_neighbor_rsu": int(val["src_neighbor_rsu"]),
            "val_src_mbs": int(val["src_mbs"]),
            "cache_overlap_avg": float(overlap_avg),
            "cache_overlap_std": float(overlap_std),
        }

        with open(log_path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=log_fields).writerow(row)

        if log_every and (upd % log_every == 0):
            tqdm.write(
                f"[LOG] upd={upd} reward={row['rollout_reward_mean']:.4f} "
                f"entropy={row['entropy_mean']:.4f} "
                f"KL={row['approx_kl']:.4f} clip={row['clipfrac']:.3f} EV={row['explained_var']:.3f} "
                f"val_hit={row['val_hit_ratio']:.4f} val_delay={row['val_avg_delay']:.4f} "
                f"overlap={row['cache_overlap_avg']:.3f}"
            )

        if checkpoint_path is not None and (upd % checkpoint_freq) == 0:
            save_gat_checkpoint(
                update_idx=upd,
                gat_encoder=gat_encoder,
                policy=policy,
                critic=critic,
                opt_actor=opt_actor,
                opt_critic=opt_critic,
                rsus=rsus,
                vehicles=vehicles,
                t_ptr=t_ptr,
                prev_ts=prev_ts,
                path=checkpoint_path,
            )

    gat_encoder.eval(); policy.eval(); critic.eval()
    return {"encoder": gat_encoder, "policy": policy, "critic": critic}


In [ ]:
from collections import Counter
import numpy as np
import torch
from tqdm.notebook import tqdm

def evaluate_gat_on_test(
    rsus, vehicles,
    test_ratings, user_id_to_idx, movie_id_to_idx,
    gat_encoder, policy,
    arrival_attn, ITEM_EMB, encoded_users,
    device=device, test_fraction=1.0,
    topk_s=64,
    return_delay_percentiles=True
):
    ts_list, ts_to_reqs = build_requests_by_timestamp(test_ratings, user_id_to_idx, movie_id_to_idx)
    if len(ts_list) == 0:
        return dict(hit_ratio=0.0, avg_delay=0.0, requests=0,
                    src_self=0, src_v2v=0, src_rsu=0, src_neighbor_rsu=0, src_mbs=0,
                    p95_delay=0.0, p99_delay=0.0)

    if test_fraction < 1.0:
        n_keep = max(1, int(len(ts_list) * test_fraction))
        ts_list = ts_list[:n_keep]

    for v in vehicles:
        v["prev_rsu_index"] = v.get("rsu_index", None)
    neighbors = rsu_neighbors_map(rsus)

    total_reqs, total_hits, delays = 0, 0, []
    prev_ts = ts_list[0]
    src_counter = Counter()

    gat_encoder.eval()
    policy.eval()

    with torch.inference_mode():
        for t in tqdm(ts_list, desc="Evaluating GAT", leave=False):
            dt = max(0, t - prev_ts); prev_ts = t
            if dt > 0:
                step_move_vehicles(vehicles, dt)
                update_vehicle_rsu_assignments(vehicles, rsus)
                handle_vehicle_arrivals(rsus, vehicles, arrival_attn, ITEM_EMB, neighbors)

            ts_requests = ts_to_reqs[t]
            tmp_batches = group_requests_by_rsu(ts_requests, build_user_to_vehicle_map(vehicles), vehicles)
            recent_local = {rid: {fid for (_, fid) in batch} for rid, batch in tmp_batches.items()}
            div_controller.update_effective_priorities(rsus, neighbors, local_recent_requests=recent_local)

            x_r, x_v, adj_rv, adj_rr = build_graph_inputs(rsus, vehicles, encoded_users, topk_s=topk_s)
            z_r = gat_encoder(x_r, x_v, adj_rv, adj_rr)

            for rid, batch in tmp_batches.items():
                rsu = rsus[rid]
                vehs_here = vehicles_under_rsu(rid, vehicles)
                if not vehs_here:
                    continue

                act = policy.act(z_r[rid], rsu)
                apply_ephemeral_inserts(rsu, act["ephemeral_ids"], RSU_CACHE_ITEMS)
                apply_persistent_swaps(rsu, act["n_swaps"], RSU_CACHE_ITEMS)

                for (veh, fid) in batch:
                    src, dly = find_source_and_delays(veh, fid, rsu, vehs_here, rsus, neighbors)
                    src_counter[src] += 1

                    if src == "self":
                        tau = 0.0; hit = True
                    elif src == "v2v":
                        tau = dly["v2v"]; hit = True
                    elif src == "rsu":
                        tau = dly["v2r"]; hit = True
                    elif src == "neighbor_rsu":
                        tau = dly["v2r"] + dly["r2r"]; hit = True
                    else:
                        tau = dly["mbs"]; hit = False

                    total_reqs += 1
                    total_hits += int(hit)
                    delays.append(tau)

    avg_delay = float(np.mean(delays)) if delays else 0.0
    p95 = float(np.percentile(delays, 95)) if (delays and return_delay_percentiles) else np.nan
    p99 = float(np.percentile(delays, 99)) if (delays and return_delay_percentiles) else np.nan

    return dict(
        hit_ratio=(total_hits / max(1, total_reqs)),
        avg_delay=avg_delay,
        p95_delay=p95,
        p99_delay=p99,
        requests=total_reqs,
        src_self=int(src_counter["self"]),
        src_v2v=int(src_counter["v2v"]),
        src_rsu=int(src_counter["rsu"]),
        src_neighbor_rsu=int(src_counter["neighbor_rsu"]),
        src_mbs=int(src_counter["mbs"]),
    )


In [ ]:
import os, csv, copy
import torch

RSU_CACHE_ITEMS = 1000
VEH_CACHE_ITEMS = 1000

hard_reset_episode(rsus, vehicles)

neighbors = rsu_neighbors_map(rsus)

n = os.cpu_count() or 8
torch.set_num_threads(n)
torch.set_num_interop_threads(max(1, n // 2))
print("torch threads:", torch.get_num_threads(), torch.get_num_interop_threads())

resume_flag = True
ckpt_load = CHECKPOINT_PATH_LOAD
if ckpt_load is None or (not os.path.isfile(ckpt_load)):
    resume_flag = False
    ckpt_load = None

trained = train_gat_ppo(
    rsus, vehicles,
    ratings_dataset=train_ratings,
    user_id_to_idx=user_id_to_idx,
    movie_id_to_idx=movie_id_to_idx,
    gat_encoder=gat_encoder,
    policy=policy,
    critic=critic,
    arrival_attn=arrival_attn,
    ITEM_EMB=ITEM_EMB,
    encoded_users=encoded_users,
    steps_per_update=64,
    minibatch_epochs=MINIBATCH_EPOCHS,
    minibatch_splits=MINIBATCH_SPLITS,
    total_updates=64,
    gamma=GAMMA, lam=LAMBDA_GAE,
    lr_actor=LR_ACTOR, lr_critic=LR_CRITIC,
    topk_s=TOPK_S,
    log_every=10,
    train_fraction=1.0,
    resume=resume_flag,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_path_load=ckpt_load,
    checkpoint_freq=5
)

snap_rsus, snap_vehicles = snapshot_state(rsus, vehicles)

metrics = evaluate_gat_on_test(
    rsus, vehicles,
    test_ratings=test_ratings,
    user_id_to_idx=user_id_to_idx,
    movie_id_to_idx=movie_id_to_idx,
    gat_encoder=trained["encoder"],
    policy=trained["policy"],
    arrival_attn=arrival_attn,
    ITEM_EMB=ITEM_EMB,
    encoded_users=encoded_users,
    device=device,
    test_fraction=1.0
)

restore_state(snap_rsus, snap_vehicles)

print("\n=== GAT-FDRL Evaluation Results ===")
print(f"Total Requests      : {metrics['requests']}")
print(f"Cache Hit Rate      : {metrics['hit_ratio']:.4f}")
print(f"Average Delay (s)   : {metrics['avg_delay']:.4f}")

print("\n--- Content Source Breakdown ---")
print(f"Self (Vehicle Cache): {metrics.get('src_self', 0)}")
print(f"V2V                 : {metrics.get('src_v2v', 0)}")
print(f"RSU                 : {metrics.get('src_rsu', 0)}")
print(f"Neighbor RSU        : {metrics.get('src_neighbor_rsu', 0)}")
print(f"MBS (Miss)          : {metrics.get('src_mbs', 0)}")

results_path = "results/results.csv"
file_exists = os.path.isfile(results_path)

row = {
    "hit_ratio": metrics["hit_ratio"],
    "avg_delay": metrics["avg_delay"],
    "requests": metrics["requests"],
    "src_self": metrics.get("src_self", 0),
    "src_v2v": metrics.get("src_v2v", 0),
    "src_rsu": metrics.get("src_rsu", 0),
    "src_neighbor_rsu": metrics.get("src_neighbor_rsu", 0),
    "src_mbs": metrics.get("src_mbs", 0),
}

with open(results_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"\nResults saved to: {results_path}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

log_path = "results/1000/train_logs.csv"
df = pd.read_csv(log_path)

df = df.sort_values("update").reset_index(drop=True)

def plot_series(x, y, title, ylabel):
    plt.figure(figsize=(8,4))
    plt.plot(x, y)
    plt.title(title)
    plt.xlabel("Update")
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()

plt.figure(figsize=(8,4))
plt.plot(df["update"], df["rollout_reward_mean"], label="mean")
if "rollout_reward_std" in df.columns:
    m = df["rollout_reward_mean"].to_numpy()
    s = df["rollout_reward_std"].to_numpy()
    plt.fill_between(df["update"], m - s, m + s, alpha=0.2, label="±1 std")
plt.title("Rollout Reward vs Update")
plt.xlabel("Update"); plt.ylabel("Reward")
plt.grid(True); plt.legend()
plt.show()

if "entropy_mean" in df.columns:
    plot_series(df["update"], df["entropy_mean"], "Policy Entropy vs Update", "Entropy")

if "value_loss_last" in df.columns:
    plot_series(df["update"], df["value_loss_last"], "Value Loss (last minibatch) vs Update", "MSE Loss")
if "policy_loss_last" in df.columns:
    plot_series(df["update"], df["policy_loss_last"], "Policy Loss (last minibatch) vs Update", "PPO Loss")

if "val_hit_ratio" in df.columns:
    d = df.dropna(subset=["val_hit_ratio"])
    if len(d):
        plot_series(d["update"], d["val_hit_ratio"], "Validation Hit Ratio vs Update", "Hit Ratio")

if "val_avg_delay" in df.columns:
    d = df.dropna(subset=["val_avg_delay"])
    if len(d):
        plot_series(d["update"], d["val_avg_delay"], "Validation Avg Delay vs Update", "Delay (s)")

for col in ["p50_delay", "p95_delay", "p99_delay", "val_p50_delay", "val_p95_delay", "val_p99_delay"]:
    if col in df.columns:
        d = df.dropna(subset=[col])
        if len(d):
            plot_series(d["update"], d[col], f"{col} vs Update", "Delay (s)")

src_cols = ["val_src_self", "val_src_v2v", "val_src_rsu", "val_src_neighbor_rsu", "val_src_mbs"]
if all(c in df.columns for c in src_cols):
    d = df.dropna(subset=src_cols).copy()
    if len(d):
        total = d[src_cols].sum(axis=1).replace(0, np.nan)
        plt.figure(figsize=(8,4))
        for c in src_cols:
            plt.plot(d["update"], d[c]/total, label=c.replace("val_src_", ""))
        plt.title("Validation Source Fractions vs Update")
        plt.xlabel("Update"); plt.ylabel("Fraction")
        plt.grid(True); plt.legend()
        plt.show()

if "cache_overlap_avg" in df.columns:
    plt.figure(figsize=(8,4))
    plt.plot(df["update"], df["cache_overlap_avg"], label="avg")
    if "cache_overlap_std" in df.columns:
        m = df["cache_overlap_avg"].to_numpy()
        s = df["cache_overlap_std"].to_numpy()
        plt.fill_between(df["update"], m - s, m + s, alpha=0.2, label="±1 std")
    plt.title("Neighbor Cache Overlap vs Update (Lower is More Diverse)")
    plt.xlabel("Update"); plt.ylabel("Jaccard overlap")
    plt.grid(True); plt.legend()
    plt.show()

if "sec_per_update" in df.columns:
    plot_series(df["update"], df["sec_per_update"], "Seconds per Update vs Update", "Seconds")
    cum = np.cumsum(df["sec_per_update"].fillna(0).to_numpy())
    plt.figure(figsize=(8,4))
    plt.plot(df["update"], cum)
    plt.title("Cumulative Training Time vs Update (Wall-clock convergence proxy)")
    plt.xlabel("Update"); plt.ylabel("Seconds (cumulative)")
    plt.grid(True)
    plt.show()


In [ ]:
class StudentGATLayer(nn.Module):
    def __init__(self, q_dim, k_dim, out_dim, n_heads=2):
        super().__init__()
        self.nh = n_heads
        self.Wq = nn.Linear(q_dim, out_dim * n_heads, bias=False)
        self.Wk = nn.Linear(k_dim, out_dim * n_heads, bias=False)
        self.Wv = nn.Linear(k_dim, out_dim * n_heads, bias=False)

    def forward(self, Q, K, nbrs_idx, return_attn=False):
        R = Q.size(0)
        q = self.Wq(Q).view(R, self.nh, -1)
        k_all = self.Wk(K)
        v_all = self.Wv(K)
        d = q.size(-1)
        outs, attns = [], []
        for r in range(R):
            idx = nbrs_idx[r]
            if not idx:
                outs.append(torch.zeros(self.nh, d, device=Q.device))
                attns.append(torch.zeros(self.nh, 1, device=Q.device))
                continue
            k = k_all[idx].view(len(idx), self.nh, d)
            v = v_all[idx].view(len(idx), self.nh, d)
            scores = (q[r][None, :, :] * k.transpose(0,1)).sum(dim=-1)  # [H, Nr]
            alpha = torch.softmax(scores, dim=-1)                        # [H, Nr]
            ctx = torch.einsum("hn,nhd->hd", alpha, v.transpose(0,1))   # [H, d]
            outs.append(ctx)
            attns.append(alpha)
        out = torch.stack(outs, dim=0).reshape(R, -1)
        if return_attn:
            return out, attns  # list length R, each [H, Nr]
        return out

class StudentRSUGATEncoder(nn.Module):
    def __init__(self, fr_rsu, fv_veh, hidden=64, heads=2, out_dim=128):
        super().__init__()
        self.veh_attn = StudentGATLayer(q_dim=fr_rsu, k_dim=fv_veh, out_dim=hidden, n_heads=heads)
        self.rsu_attn = StudentGATLayer(q_dim=fr_rsu, k_dim=fr_rsu, out_dim=hidden, n_heads=heads)
        self.proj = nn.Sequential(
            nn.Linear(fr_rsu + hidden*heads + hidden*heads, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
        self.out_dim = out_dim

    def forward(self, x_r, x_v, adj_rv, adj_rr, return_attn=False):
        x_r = x_r.to(device); x_v = x_v.to(device)
        veh_ctx, veh_alpha = self.veh_attn(x_r, x_v, adj_rv, return_attn=True)
        rsu_ctx, rsu_alpha = self.rsu_attn(x_r, x_r, adj_rr, return_attn=True)
        z = torch.cat([x_r, veh_ctx, rsu_ctx], dim=1)
        z = self.proj(z)
        if return_attn:
            return z, veh_alpha, rsu_alpha
        return z

class StudentPolicy(nn.Module):
    def __init__(self, z_dim=128, max_swaps=4):
        super().__init__()
        self.max_swaps = max_swaps
        self.actor = nn.Sequential(
            nn.Linear(z_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, max_swaps + 1)
        )
    def forward(self, z_r):
        return self.actor(z_r)  # logits [R, M+1]

In [ ]:
class KDDataset:
    def __init__(self):
        self.records = []

    def add(self, **kv):
        rec = {}
        for k, v in kv.items():
            if isinstance(v, torch.Tensor):
                rec[k] = v.detach().cpu()
            else:
                rec[k] = v
        self.records.append(rec)

    def __len__(self): return len(self.records)

def collect_kd_rollouts(
    rsus, vehicles, ratings_dataset, user_id_to_idx, movie_id_to_idx,
    teacher_encoder, teacher_policy, teacher_critic=None,
    max_steps=2000, topk_s=64
):
    teacher_encoder.eval(); teacher_policy.eval()
    dataset = KDDataset()
    ts_list, ts_to_reqs = build_requests_by_timestamp(ratings_dataset, user_id_to_idx, movie_id_to_idx)
    assert len(ts_list) > 0

    for v in vehicles: v["prev_rsu_index"] = v.get("rsu_index", None)
    neighbors = rsu_neighbors_map(rsus)

    t_ptr, prev_ts, steps = 0, ts_list[0], 0
    with torch.no_grad():
        while steps < max_steps:
            if t_ptr >= len(ts_list):
                t_ptr = 0; prev_ts = ts_list[0]
            t = ts_list[t_ptr]
            dt = max(0, t - prev_ts); prev_ts = t
            t_ptr += 1

            if dt > 0:
                step_move_vehicles(vehicles, dt)
                update_vehicle_rsu_assignments(vehicles, rsus)
                handle_vehicle_arrivals(rsus, vehicles, arrival_attn, ITEM_EMB, neighbors)

            ts_requests = ts_to_reqs[t]
            tmp_batches = group_requests_by_rsu(ts_requests, build_user_to_vehicle_map(vehicles), vehicles)
            recent_local = {rid: {fid for (_, fid) in batch} for rid, batch in tmp_batches.items()}
            div_controller.update_effective_priorities(rsus, neighbors, local_recent_requests=recent_local)

            x_r, x_v, adj_rv, adj_rr = build_graph_inputs(rsus, vehicles, encoded_users, topk_s=topk_s)
            z_r = teacher_encoder(x_r, x_v, adj_rv, adj_rr)                 # [R, Zt]
            logits = policy.actor(z_r) if hasattr(teacher_policy, "actor") else teacher_policy(z_r)
            values = teacher_critic(z_r) if teacher_critic is not None else None

            dataset.add(
                x_r=x_r, x_v=x_v, adj_rv=adj_rv, adj_rr=adj_rr,
                z_teacher=z_r, logits_teacher=logits, values_teacher=values,
                alpha_rv=None, alpha_rr=None
            )

            dists = Categorical(logits=logits)
            a_swaps = dists.sample()
            for rid, batch in tmp_batches.items():
                rsu = rsus[rid]
                vehs_here = vehicles_under_rsu(rid, vehicles)
                apply_ephemeral_inserts(rsu, rsu.get("arrival_buffer", [])[:8], RSU_CACHE_ITEMS)
                apply_persistent_swaps(rsu, int(a_swaps[rid].item()), RSU_CACHE_ITEMS)

            steps += 1

    return dataset

In [ ]:
class StudentValueHead(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, z_r): return self.net(z_r).squeeze(-1)

class TeacherToStudentAlign(nn.Module):
    """Linear projector to map teacher z-dim -> student z-dim for L2 feature distillation."""
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=False)
    def forward(self, z_t):
        return self.proj(z_t)

def kd_train_student(
    kd_dataset, student_encoder, student_policy, student_value=None,
    teacher_z_dim=256, kd_epochs=5, batch_size=8,
    alpha_kl=1.0, alpha_val=0.2, alpha_feat=0.5, alpha_attn=0.0,
    lr=3e-4
):
    student_encoder.to(device).train()
    student_policy.to(device).train()
    if student_value is not None: student_value.to(device).train()

    projector = TeacherToStudentAlign(d_in=teacher_z_dim, d_out=student_encoder.out_dim).to(device)
    params = list(student_encoder.parameters()) + list(student_policy.parameters()) + list(projector.parameters())
    if student_value is not None:
        params += list(student_value.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    def kl_divergence(logits_t, logits_s, T=1.0):
        pt = torch.softmax(logits_t / T, dim=-1)
        log_ps = torch.log_softmax(logits_s / T, dim=-1)
        return torch.mean(torch.sum(pt * (torch.log(pt + 1e-8) - log_ps), dim=-1)) * (T*T)

    N = len(kd_dataset.records)
    order = np.arange(N)
    for epoch in range(1, kd_epochs+1):
        np.random.shuffle(order)
        running = 0.0
        for i in range(0, N, batch_size):
            batch_idx = order[i:i+batch_size]
            z_s_list, z_t_list, logit_t_list, logit_s_list, val_t_list, val_s_list = [], [], [], [], [], []
            feat_loss = 0.0; kl_loss = 0.0; val_loss = 0.0; attn_loss = 0.0

            opt.zero_grad()
            for bi in batch_idx:
                rec = kd_dataset.records[bi]
                x_r = rec["x_r"].to(device); x_v = rec["x_v"].to(device)
                adj_rv, adj_rr = rec["adj_rv"], rec["adj_rr"]
                z_t = rec["z_teacher"].to(device)                       # [R, Zt]
                logits_t = rec["logits_teacher"].to(device)             # [R, A]
                values_t = rec["values_teacher"].to(device) if rec["values_teacher"] is not None else None

                z_s = student_encoder(x_r, x_v, adj_rv, adj_rr)         # [R, Zs]
                logits_s = student_policy(z_s)                           # [R, A]

                kl_loss += kl_divergence(logits_t, logits_s)
                z_t_proj = projector(z_t)
                feat_loss += F.mse_loss(z_s, z_t_proj)

                if student_value is not None and values_t is not None:
                    values_s = student_value(z_s)
                    val_loss += F.mse_loss(values_s, values_t)

                if "alpha_rv" in rec and rec["alpha_rv"] is not None and alpha_attn > 0.0:
                    z_s2, s_alpha_rv, s_alpha_rr = student_encoder(x_r, x_v, adj_rv, adj_rr, return_attn=True)
                    def attn_mse(a_list_s, a_list_t):
                        loss = 0.0; count=0
                        for a_s, a_t in zip(a_list_s, a_list_t):
                            # pad to same length if necessary
                            Ls, Lt = a_s.size(1), len(a_t[0])  # [H, Nr]
                            if Ls == 0 or Lt == 0:
                                continue
                            # convert teacher list->tensor (H, Nr)
                            a_ten = torch.tensor(a_t, device=device)  # (H, Nr_t)
                            # truncate/pad to min length
                            L = min(Ls, a_ten.size(1))
                            loss += F.mse_loss(a_s[:, :L], a_ten[:, :L].float())
                            count += 1
                        return loss / max(1, count)
                    pass

            total = alpha_kl * kl_loss + alpha_feat * feat_loss + alpha_val * val_loss + alpha_attn * attn_loss
            total.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()

            running += total.item()

        print(f"[KD] epoch {epoch}/{kd_epochs} | loss {running/ math.ceil(N/max(1,batch_size)) :.4f}")

    student_encoder.eval(); student_policy.eval()
    if student_value is not None: student_value.eval()
    return {"encoder": student_encoder, "policy": student_policy, "value": student_value}

In [ ]:
hard_reset_episode(rsus, vehicles)

kd_data = collect_kd_rollouts(
    rsus, vehicles,
    ratings_dataset=train_ratings,
    user_id_to_idx=user_id_to_idx,
    movie_id_to_idx=movie_id_to_idx,
    teacher_encoder=gat_encoder,
    teacher_policy=policy,
    teacher_critic=None,
    max_steps=2000,
    topk_s=TOPK_S
)

student_encoder = StudentRSUGATEncoder(fr_rsu=TOPK_S, fv_veh=encoded_users.shape[1], hidden=64, heads=2, out_dim=128).to(device)
student_policy  = StudentPolicy(z_dim=128, max_swaps=MAX_PERSISTENT_SWAPS).to(device)
student_value   = None 

distilled = kd_train_student(
    kd_dataset=kd_data,
    student_encoder=student_encoder,
    student_policy=student_policy,
    student_value=student_value,
    teacher_z_dim=256,
    kd_epochs=5, batch_size=8,
    alpha_kl=1.0, alpha_val=0.0, alpha_feat=0.5, alpha_attn=0.0,
    lr=3e-4
)

hard_reset_episode(rsus, vehicles)

print("=== Teacher on test ===")
teacher_metrics = evaluate_gat_on_test(
    rsus, vehicles,
    test_ratings=test_ratings,
    user_id_to_idx=user_id_to_idx,
    movie_id_to_idx=movie_id_to_idx,
    gat_encoder=gat_encoder, policy=policy, device=device
)
print(teacher_metrics)

hard_reset_episode(rsus, vehicles)

print("=== Student on test ===")
student_metrics = evaluate_gat_on_test(
    rsus, vehicles,
    test_ratings=test_ratings,
    user_id_to_idx=user_id_to_idx,
    movie_id_to_idx=movie_id_to_idx,
    gat_encoder=distilled["encoder"], policy=distilled["policy"], device=device
)
print(student_metrics)